# Multi-scale audit (Track B0–B2, `future_plan.md` §18.10): does sub-cellular, patch or typed-edge structure add to the cell table?

**Inputs:**
- `pixel-features` (per-cell pixel features and image-derived cell means);
- the prep kernels (labels, positions, contacts, frozen Danenberg split).
- Every intensity comes from the same masks.

**B0, homophily:**
- per coarse class, the same-type share among labelled contact neighbours and 30 µm neighbours, against a within-image label-permutation null;
- the Table + nbr typing error rate, binned by that share.

**Base features** (as `cell-type-graph`): the 15 inputs, i.e. the 14 channels as arcsinh(cell mean / 5) plus log area, harmonised per cohort. Neighbour aggregates over kNN-10 (mean, max, SD, 2-hop mean), plus 30 µm and contact counts and mean neighbour distance. **For imputation, the target channel is removed from every block.**

**Arms** (LightGBM; all pixel / band / patch features harmonised per cohort):

| Arm | Imputation (per marker, other channels only) | Typing |
|---|---|---|
| 0 base | Table + nbr | Table + nbr |
| 1 +pix | + pixel features, full mask (dio, sd, Moran's I per channel; shape) | same |
| 2 +pix e2 | + the pixel features on the 2-px-eroded mask (spillover control 1) | same |
| 3 +band | + the outside 2-px band means (control base) | same |
| 4 +band +pix | arm 3 + full-mask pixel features (spillover control 2: pixel gain beyond the band) | same |
| 5 +B2 | + patch nodes (64 / 128 µm grid means and densities) + typed edges (contact vs non-contact 30 µm neighbour means) | + the same, + out-of-fold predicted coarse-type mixtures (contact, non-contact, 64 / 128 µm patches) and contact counts by predicted type |

**Settings:**
- imputation: train J → test J and D; train D → test D and J (11 markers);
- typing: coarse labels, trained on J and on D, each tested on both cohorts; fine labels D→D and J→J.

**Splits and seeds:** as `cell-type-graph` / `train-test-grid` (Jackson 70/10/20, redrawn per replicate; the frozen Danenberg test patients). 3 replicates.

**Scores:**
- imputation: R4 (per-cell r with the image mean removed);
- typing: macro-F1;
- 1000× patient bootstrap, paired between arms.

**Decision rules.** §18.10 fixed the thresholds. The control definitions below were set before any results:
- **B1 passes** in a setting if either:
  - (imputation) ≥ 2 markers (ERα excluded) satisfy all of:
    - Δ(1 − 0) has mean ≥ 0.02 and CI > 0 in 3/3 replicates;
    - Δ(2 − 0) has CI > 0 in 3/3;
    - Δ(4 − 3) has CI > 0 in 3/3;
  - (typing) Δ macro-F1 (1 − 0) has mean ≥ 0.01 and CI > 0 in 3/3, with the same two controls.
- **B2 passes** if either:
  - typing on J→D or D→J: Δ(5 − 0) has mean ≥ 0.01 and CI > 0 in 3/3;
  - imputation: Δ(5 − 0) has mean ≥ 0.02 and CI > 0 in 3/3 for ≥ 2 markers, in any setting.
- **B3 is built only if B1 or B2 passes.**
- SAGE+ with typed edges (the second half of B2) is deferred. It is run only if LightGBM-B2 shows a partial signal.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013                                   # same seed and split rules as cell-type-graph / train-test-grid
N_BOOT = 100 if SMOKE else 1000
COF = 5.0
N_FIT_IMP = 10_000 if SMOKE else 100_000          # training cells per imputation model
N_FIT_TYP = 10_000 if SMOKE else 150_000          # labelled training cells per typing model
N_FIT_S1 = 5_000 if SMOKE else 100_000            # stage-1 (predicted-type) models
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
BUDGET_MIN = float(os.environ.get("BUDGET_MIN", 660))
REPS = [0] if SMOKE else [0, 1, 2]
R_CAP = 32
NUC = ["histone_h3", "dna1", "dna2"]
MARKERS = sorted(["c_erb_b_2_her2", "cd20", "cd3", "cd45", "cd68", "cytokeratin_5", "cytokeratin_8_18",
                  "estrogen_receptor_alpha", "ki_67", "pan_cytokeratin", "sma"])
IN_SET = NUC + MARKERS                            # 14 channels; + log area = 15 base inputs
COARSE = ["epithelial", "T cell", "B cell", "myeloid", "endothelial", "stroma"]
D_COARSE = {"CD8^{+} T cells": "T cell", "CD4^{+} T cells": "T cell", "T_{Reg} & T_{Ex}": "T cell",
            "B cells": "B cell",
            "Macrophages": "myeloid", "Granulocytes": "myeloid", "Macrophages & granulocytes": "myeloid",
            "Endothelial": "endothelial",
            "Fibroblasts": "stroma", "Fibroblasts FSP1^{+}": "stroma", "Myofibroblasts": "stroma",
            "Myofibroblasts PDPN^{+}": "stroma"}
J_COARSE = {1: "T cell", 2: "B cell", 4: "myeloid", 6: "myeloid", 7: "endothelial",
            **{m: "stroma" for m in range(8, 14)}, **{m: "epithelial" for m in range(14, 28)}}
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

## Cells, labels and features

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "Danenberg" if "metabric_id" in pq.read_schema(p).names else "Jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p),
                  pix=g(f"pixfeat_{key.lower()}.parquet")[0])
P["Danenberg"]["split"] = near("danenberg_patient_split.csv", P["Danenberg"]["cells"])
print(json.dumps(P, indent=1))

def harm(X):
    X = np.asarray(X, np.float64)
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

def load(key):
    pf = pd.read_parquet(P[key]["pix"])
    cells = pd.read_parquet(P[key]["cells"])
    cells = cells.set_index(["image_name", "object_number"]).loc[
        pd.MultiIndex.from_arrays([pf.image_name.values, pf.object_number.values])].reset_index()
    n = len(cells)
    img, img_names = pd.factorize(cells.image_name, sort=True)
    assert (np.diff(img) >= 0).all()
    starts = np.r_[0, np.flatnonzero(np.diff(img)) + 1, n]
    loc = pd.Series(np.arange(n), index=pd.MultiIndex.from_arrays([cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    CON = sp.coo_matrix((np.ones(len(ca), np.float32), (ca, cb)), shape=(n, n)).tocsr()
    CON = ((CON + CON.T) > 0).astype(np.float32).tocsr()
    phen = cells.cell_phenotype.astype(str).values
    fine = sorted(set(phen))
    y_fine = pd.Series(phen).map({p: i for i, p in enumerate(fine)}).values.astype(np.int64)
    if key == "Danenberg":
        epi = cells.groupby(cells.cell_phenotype.astype(str)).is_epithelial.mean()
        cmap = {p: ("epithelial" if epi[p] > 0.5 else D_COARSE.get(p)) for p in fine}
        pid = "metabric_id"
    else:
        mc = cells.groupby(cells.cell_phenotype.astype(str)).metacluster.first()
        cmap = {p: J_COARSE.get(int(mc[p])) for p in fine}
        pid = "patient_id"
    coarse_of = np.array([COARSE.index(cmap[p]) if cmap[p] else -1 for p in fine], np.int64)
    X15 = np.c_[np.arcsinh(pf[[f"raw__{c}" for c in IN_SET]].values.astype(np.float64) / COF),
                np.log(np.maximum(cells["area"].values, 1))]
    F = {"base": np.nan_to_num(harm(X15))}
    pix = lambda v: [f"{s}_{v}__{c}" for c in IN_SET for s in ("dio", "sd", "mi")]
    shape = ["ecc", "perim", "dmax"]
    F["pix"] = harm(pf[pix("full") + shape].values)
    F["pix_e2"] = harm(pf[pix("e2") + shape].values)
    F["band"] = harm(pf[[f"band__{c}" for c in IN_SET]].values)
    names = {"pix": pix("full") + shape, "pix_e2": pix("e2") + shape, "band": [f"band__{c}" for c in IN_SET]}
    return dict(key=key, n=n, starts=starts, img=img.astype(np.int64), n_img=len(img_names), img_names=np.asarray(img_names),
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                CON=CON, fine=fine, y={"fine": y_fine, "coarse": coarse_of[y_fine]}, F=F, names=names)

CO = {k: load(k) for k in ("Jackson", "Danenberg")}
CJ, CD = CO["Jackson"], CO["Danenberg"]
split0 = pd.read_csv(P["Danenberg"]["split"]).set_index("metabric_id").split
CD["split0"] = pd.Series(CD["patient"]).map(split0).fillna("none").values
for k, C in CO.items():
    print(f"{k}: {C['n']:,} cells, {C['n_img']} images, {len(set(C['patient']))} patients; "
          f"pixel-feature NaN fraction {np.isnan(C['F']['pix']).mean():.3f} (full) / {np.isnan(C['F']['pix_e2']).mean():.3f} (e2)")
tick("tables and pixel features loaded")

## Graphs: kNN-10 neighbour block, 30 µm contact / non-contact split, patch grids

In [ ]:
def adjacency(C, kind, par):
    rows, cols = [], []
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        xy = C["xy"][a:b]
        k = min(par if kind == "knn" else R_CAP, b - a - 1)
        if kind == "knn":
            d, nb = cKDTree(xy).query(xy, k=k + 1)
        else:
            d, nb = cKDTree(xy).query(xy, k=k + 1, distance_upper_bound=par)
        d, nb = d[:, 1:], nb[:, 1:]
        ok = np.isfinite(d)
        rows.append(np.repeat(np.arange(a, b), k)[ok.ravel()])
        cols.append(nb[ok] + a)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    A = sp.coo_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(C["n"], C["n"])).tocsr()
    A = ((A + A.T) > 0).astype(np.float32).tocsr()
    A.sort_indices()
    return A

def row_mean(A, X):
    deg = np.asarray(A.sum(1)).ravel()
    return np.where(deg[:, None] > 0, (A @ X) / np.maximum(deg, 1)[:, None], np.nan).astype(np.float32), deg

def grid_mean(C, X, size):
    """Mean of X over the cells in the same size-µm grid square of the same image, and the square's cell count."""
    gx = np.floor(C["xy"][:, 0] / size).astype(np.int64)
    gy = np.floor(C["xy"][:, 1] / size).astype(np.int64)
    key = (C["img"] * 100_000 + gy) * 100_000 + gx
    code, inv = np.unique(key, return_inverse=True)
    cnt = np.bincount(inv).astype(np.float64)
    M = np.stack([np.bincount(inv, X[:, j], len(code)) for j in range(X.shape[1])], 1) / cnt[:, None]
    return M[inv].astype(np.float32), np.log1p(cnt[inv]).astype(np.float32)

for key, C in CO.items():
    X = C["F"]["base"]
    A10 = adjacency(C, "knn", 10)
    A30 = adjacency(C, "radius", 30.0)
    deg = np.diff(A10.indptr).astype(np.float32)[:, None]
    mean = (A10 @ X) / deg
    std = np.sqrt(np.maximum((A10 @ (X * X)) / deg - mean * mean, 0))
    mx = np.maximum.reduceat(X[A10.indices], A10.indptr[:-1], axis=0)
    hop2 = (A10 @ mean) / deg
    dst = np.repeat(np.arange(C["n"]), np.diff(A10.indptr))
    dist = np.bincount(dst, np.linalg.norm(C["xy"][A10.indices] - C["xy"][dst], axis=1), C["n"]) / deg.ravel()
    C["F"]["nbr_blocks"] = np.stack([mean, mx, std, hop2], 0).astype(np.float32)          # 4 x n x 15
    C["F"]["nbr_extra"] = np.c_[np.log1p(np.diff(A30.indptr)), np.log1p(np.diff(C["CON"].indptr)), dist / 30.0].astype(np.float32)
    CON30 = A30.multiply(C["CON"]).tocsr()
    NON30 = (A30 - CON30).tocsr()
    NON30.eliminate_zeros()
    C["A_con"], C["A_non"] = C["CON"], NON30
    m_con, _ = row_mean(C["CON"], X)
    m_non, _ = row_mean(NON30, X)
    g64, n64 = grid_mean(C, X, 64.0)
    g128, n128 = grid_mean(C, X, 128.0)
    C["F"]["b2_blocks"] = np.stack([m_con, m_non, g64, g128], 0)                            # 4 x n x 15
    C["F"]["b2_extra"] = np.c_[n64, n128]
    del A10, A30, CON30, mean, std, mx, hop2
    gc.collect()
    print(f"{key}: kNN-10 / 30 µm / contact / non-contact graphs built; mean contact degree {np.diff(C['CON'].indptr).mean():.2f}")
tick("graphs and neighbour features")

def table_nbr(C, drop=None):
    """Table + nbr (cell-type-graph's 78 inputs); drop = base column index to remove from every block (imputation)."""
    keep = [j for j in range(15) if j != drop]
    B = C["F"]["nbr_blocks"]
    return np.concatenate([C["F"]["base"][:, keep]] + [B[i][:, keep] for i in range(4)] + [C["F"]["nbr_extra"]], 1)

def b2_safe(C, drop=None):
    keep = [j for j in range(15) if j != drop]
    B = C["F"]["b2_blocks"]
    return np.concatenate([B[i][:, keep] for i in range(4)] + [C["F"]["b2_extra"]], 1)

def pix_cols(C, kind, drop_chan=None):
    """Pixel (or band) feature columns, without the target channel's columns."""
    names = C["names"][kind]
    keep = [i for i, nm in enumerate(names) if drop_chan is None or not nm.endswith(f"__{drop_chan}")]
    return C["F"][kind][:, keep]

## Scoring helpers (R4 for imputation; macro-F1 for typing; patient bootstrap)

In [ ]:
def r4_scorer(C, te, seed):
    tp, tcode = np.unique(C["patient"][te], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    img = C["img"][te]
    def f(a, b):
        ok = np.isfinite(a) & np.isfinite(b)
        a, b, im, pc = a[ok].astype(np.float64), b[ok].astype(np.float64), img[ok], tcode[ok]
        k = int(img.max()) + 1
        cnt = np.maximum(np.bincount(im, minlength=k), 1)
        a = a - (np.bincount(im, a, k) / cnt)[im]
        b = b - (np.bincount(im, b, k) / cnt)[im]
        sxy, sxx, syy = (np.bincount(pc, a * b, len(tp)), np.bincount(pc, a * a, len(tp)), np.bincount(pc, b * b, len(tp)))
        return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (W @ sxy) / np.sqrt((W @ sxx) * (W @ syy))
    return f

def f1_scorer(C, imgs_cells, task, ncls, seed):
    node = imgs_cells[C["y"][task][imgs_cells] >= 0]
    pats, pc = np.unique(C["patient"][node], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
    y = C["y"][task][node]
    present = np.bincount(y, minlength=ncls) > 0
    def metrics(M):
        M = M.reshape(-1, ncls, ncls)
        tp_ = np.einsum("bii->bi", M)
        sup, prd = M.sum(2), M.sum(1)
        with np.errstate(invalid="ignore", divide="ignore"):
            f1 = np.where(sup + prd > 0, 2 * tp_ / (sup + prd), 0.0)
        return f1[:, present].mean(1), f1
    def f(pred_node):
        cm = np.bincount((pc * ncls + y) * ncls + pred_node, minlength=len(pats) * ncls * ncls).reshape(len(pats), ncls * ncls).astype(np.float64)
        pt, f1pt = metrics(cm.sum(0, keepdims=True))
        bt, _ = metrics(W @ cm)
        return float(pt[0]), bt, f1pt[0]
    return node, f

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def cells_of(C, imgs):
    st = C["starts"]
    return np.concatenate([np.arange(st[i], st[i + 1]) for i in imgs]) if len(imgs) else np.zeros(0, np.int64)

D_TRVA_PAT = np.unique(CD["patient"][CD["split0"] != "test"])

def make_splits(REP):
    jp, jcode = np.unique(CJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(CD["split0"] == "test", "test", np.where(pd.Series(CD["patient"]).isin(dval).values, "val", "train"))
    out = {}
    for k, s in (("Jackson", J_SPLIT), ("Danenberg", D_SPLIT)):
        C = CO[k]
        si = s[C["starts"][:-1]]
        out[k] = {t: np.flatnonzero(si == t) for t in ("train", "val", "test")}
    return out

## B0: homophily (replicate-0 splits; labels only)

In [ ]:
B0_ROWS = []
for k, C in CO.items():
    y = C["y"]["coarse"]
    lab = y >= 0
    for gname, A in (("contact", C["CON"]), ("30um_noncontact", C["A_non"])):
        rows = np.repeat(np.arange(C["n"]), np.diff(A.indptr))
        cols = A.indices
        ok = lab[rows] & lab[cols]
        same = (y[rows] == y[cols])[ok]
        r_ok = rows[ok]
        n_lab = np.bincount(r_ok, minlength=C["n"])
        share = np.bincount(r_ok, same.astype(np.float64), C["n"]) / np.maximum(n_lab, 1)
        C.setdefault("share", {})[gname] = np.where(n_lab > 0, share, np.nan)
        # null: permute labels within each image (5 permutations)
        null = []
        rs = np.random.default_rng(SEED + 99)
        for _ in range(5):
            yp = y.copy()
            for a, b in zip(C["starts"][:-1], C["starts"][1:]):
                seg = yp[a:b]
                m = seg >= 0
                seg[m] = rs.permutation(seg[m])
                yp[a:b] = seg
            sp_ = (yp[rows] == yp[cols])[ok]
            null.append(np.bincount(y[r_ok], sp_.astype(np.float64), len(COARSE)) / np.maximum(np.bincount(y[r_ok], minlength=len(COARSE)), 1))
        obs = np.bincount(y[r_ok], same.astype(np.float64), len(COARSE)) / np.maximum(np.bincount(y[r_ok], minlength=len(COARSE)), 1)
        nullm = np.mean(null, 0)
        for c in range(len(COARSE)):
            B0_ROWS.append(dict(cohort=k, graph=gname, cls=COARSE[c], edges=int((y[r_ok] == c).sum()),
                                same_type_share=float(obs[c]), null_share=float(nullm[c]), ratio=float(obs[c] / max(nullm[c], 1e-9))))
B0 = pd.DataFrame(B0_ROWS)
B0.to_csv(f"{OUT}/b0_homophily.csv", index=False)
print(B0.round(3).to_string(index=False))
tick("B0 homophily")

## Stage-1 predicted coarse types (for the B2 typing features), per training cohort and replicate

In [ ]:
def class_weights(y, ncls):
    cnt = np.bincount(y, minlength=ncls).astype(np.float64)
    w = np.where(cnt > 0, 1 / np.sqrt(np.maximum(cnt, 1)), 0.0)
    return w / w[y].mean()

def fit_clf(X, y, ncls, seed, n_est=None):
    prm = dict(LGB)
    if n_est:
        prm["n_estimators"] = n_est
    return lgb.LGBMClassifier(random_state=seed, **prm).fit(X, y, sample_weight=class_weights(y, ncls)[y])

def proba_full(mdl, X, ncls):
    P_ = np.zeros((len(X), ncls), np.float32)
    for s in range(0, len(X), 200_000):
        P_[s:s + 200_000][:, mdl.classes_] = mdl.predict_proba(X[s:s + 200_000])
    return P_

def stage1(src, SPL, seed):
    """Out-of-fold (5 folds by image) coarse-type probabilities on the source training images; a full model elsewhere."""
    C = CO[src]
    XT = {k: table_nbr(CO[k]) for k in CO}
    tr_imgs = SPL[src]["train"]
    folds = np.random.default_rng(seed).permutation(len(tr_imgs)) % 5
    rs = np.random.default_rng(seed + 1)
    PR = {k: np.zeros((CO[k]["n"], len(COARSE)), np.float32) for k in CO}
    def fit_on(imgs):
        idx = cells_of(C, imgs)
        idx = idx[C["y"]["coarse"][idx] >= 0]
        idx = np.sort(rs.choice(idx, min(N_FIT_S1, len(idx)), replace=False))
        return fit_clf(XT[src][idx], C["y"]["coarse"][idx], len(COARSE), seed, n_est=60 if SMOKE else 150)
    for f in range(5):
        mdl = fit_on(tr_imgs[folds != f])
        idx = cells_of(C, tr_imgs[folds == f])
        PR[src][idx] = proba_full(mdl, XT[src][idx], len(COARSE))
    mdl = fit_on(tr_imgs)
    for k in CO:
        other = np.setdiff1d(np.arange(CO[k]["n_img"]), tr_imgs) if k == src else np.arange(CO[k]["n_img"])
        idx = cells_of(CO[k], other)
        PR[k][idx] = proba_full(mdl, XT[k][idx], len(COARSE))
    del XT
    return PR

def b2_pred(C, PRk):
    """Predicted-type mixtures over contact / non-contact neighbours and 64 / 128 µm patches, + contact counts by predicted type."""
    m_con, _ = row_mean(C["A_con"], PRk)
    m_non, _ = row_mean(C["A_non"], PRk)
    g64, _ = grid_mean(C, PRk, 64.0)
    g128, _ = grid_mean(C, PRk, 128.0)
    hard = np.eye(len(COARSE), dtype=np.float32)[PRk.argmax(1)]
    cnt_con = C["A_con"] @ hard
    return np.c_[m_con, m_non, g64, g128, np.log1p(cnt_con)].astype(np.float32)

## Replicates: imputation and typing arms

In [ ]:
IMP_ARMS = ["0 base", "1 +pix", "2 +pix e2", "3 +band", "4 +band +pix", "5 +B2"]
TYP_ARMS = IMP_ARMS
TYP_UNITS = [("coarse", "Jackson"), ("coarse", "Danenberg"), ("fine", "Danenberg"), ("fine", "Jackson")]
OTHER = {"Jackson": "Danenberg", "Danenberg": "Jackson"}

def imp_features(C, arm, k_base, chan):
    base = table_nbr(C, drop=k_base)
    if arm == "0 base":
        return base
    if arm == "1 +pix":
        return np.c_[base, pix_cols(C, "pix", chan)]
    if arm == "2 +pix e2":
        return np.c_[base, pix_cols(C, "pix_e2", chan)]
    if arm == "3 +band":
        return np.c_[base, pix_cols(C, "band", chan)]
    if arm == "4 +band +pix":
        return np.c_[base, pix_cols(C, "band", chan), pix_cols(C, "pix", chan)]
    if arm == "5 +B2":
        return np.c_[base, b2_safe(C, drop=k_base)]

def typ_features(C, arm, PRk):
    base = table_nbr(C)
    if arm == "0 base":
        return base
    if arm == "1 +pix":
        return np.c_[base, C["F"]["pix"]]
    if arm == "2 +pix e2":
        return np.c_[base, C["F"]["pix_e2"]]
    if arm == "3 +band":
        return np.c_[base, C["F"]["band"]]
    if arm == "4 +band +pix":
        return np.c_[base, C["F"]["band"], C["F"]["pix"]]
    if arm == "5 +B2":
        return np.c_[base, b2_safe(C), b2_pred(C, PRk)]

IMP_ROWS, IMP_DIFF, TYP_ROWS, TYP_DIFF, TYP_PCL, LOG, SKIPPED = [], [], [], [], [], [], []
EST = {}
for REP in REPS:
    t_rep = time.time()
    if REP != REPS[0] and (time.time() - T0) / 60 + EST.get("rep", 200) > BUDGET_MIN:
        print(f"!!! skipping replicate {REP}")
        SKIPPED.append(("rep", REP))
        continue
    tick(f"replicate {REP}")
    SPL = make_splits(REP)
    TE = {k: cells_of(CO[k], SPL[k]["test"]) for k in CO}
    # ---------------- imputation ----------------
    R4S = {k: r4_scorer(CO[k], TE[k], SEED + 31 + REP + 1000 * (k == "Danenberg")) for k in CO}
    for src in ("Jackson", "Danenberg"):
        C = CO[src]
        tr = cells_of(C, SPL[src]["train"])
        for m in MARKERS:
            kb = IN_SET.index(m)
            y_all = C["F"]["base"][:, kb]
            fit = np.sort(np.random.default_rng(SEED + REP).choice(tr, min(N_FIT_IMP, len(tr)), replace=False))
            BOOT = {}
            for arm in IMP_ARMS:
                t0 = time.time()
                Xf = imp_features(C, arm, kb, m)
                mdl = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(Xf[fit], y_all[fit])
                del Xf
                for test in (src, OTHER[src]):
                    Ct = CO[test]
                    pred = mdl.predict(imp_features(Ct, arm, kb, m)[TE[test]])
                    pt, bt = R4S[test](Ct["F"]["base"][TE[test], kb], pred)
                    BOOT[(arm, test)] = (pt, bt)
                    lo, hi = ci(bt)
                    IMP_ROWS.append(dict(rep=REP, train=src, test=test, marker=m, arm=arm, R4=pt, lo=lo, hi=hi))
                LOG.append(dict(rep=REP, block="imputation", train=src, marker=m, arm=arm, seconds=time.time() - t0))
            for test in (src, OTHER[src]):
                for a, b in (("1 +pix", "0 base"), ("2 +pix e2", "0 base"), ("4 +band +pix", "3 +band"),
                             ("3 +band", "0 base"), ("5 +B2", "0 base")):
                    lo, hi = ci(BOOT[(a, test)][1] - BOOT[(b, test)][1])
                    IMP_DIFF.append(dict(rep=REP, train=src, test=test, marker=m, arm=a, base=b,
                                         diff=BOOT[(a, test)][0] - BOOT[(b, test)][0], lo=lo, hi=hi))
        tick(f"rep {REP} imputation, train {src}")
        pd.DataFrame(IMP_ROWS).to_csv(f"{OUT}/imputation_scores.csv", index=False)
        pd.DataFrame(IMP_DIFF).to_csv(f"{OUT}/imputation_diffs.csv", index=False)
    # ---------------- typing ----------------
    PR = {src: stage1(src, SPL, SEED + 61 + REP) for src in ("Jackson", "Danenberg")}
    tick(f"rep {REP} stage-1 predicted types")
    for task, src in TYP_UNITS:
        el = (time.time() - T0) / 60
        if el + EST.get((task, src), 20) > BUDGET_MIN:
            print(f"!!! skipping rep {REP} {task} {src}")
            SKIPPED.append((REP, task, src))
            continue
        t_u = time.time()
        C = CO[src]
        ncls = len(COARSE) if task == "coarse" else len(C["fine"])
        tests = [src, OTHER[src]] if task == "coarse" else [src]
        idx = cells_of(C, SPL[src]["train"])
        idx = idx[C["y"][task][idx] >= 0]
        fit = np.sort(np.random.default_rng(SEED + REP).choice(idx, min(N_FIT_TYP, len(idx)), replace=False))
        SC = {t: f1_scorer(CO[t], TE[t], task, ncls, SEED + 41 + REP + 1000 * (t == "Danenberg")) for t in tests}
        BOOT = {}
        for arm in TYP_ARMS:
            t0 = time.time()
            Xf = typ_features(C, arm, PR[src][src])
            mdl = fit_clf(Xf[fit], C["y"][task][fit], ncls, SEED + REP)
            del Xf
            for t in tests:
                node, f = SC[t]
                Xt = typ_features(CO[t], arm, PR[src][t])[node]
                pred = mdl.classes_[mdl.predict_proba(Xt).argmax(1)]
                pt, bt, f1c = f(pred)
                BOOT[(arm, t)] = (pt, bt)
                lo, hi = ci(bt)
                TYP_ROWS.append(dict(rep=REP, task=task, train=src, test=t, arm=arm, macro_f1=pt, lo=lo, hi=hi))
                labs = COARSE if task == "coarse" else CO[t]["fine"]
                for c in range(ncls):
                    TYP_PCL.append(dict(rep=REP, task=task, train=src, test=t, arm=arm, label=labs[c], f1=float(f1c[c])))
                if REP == REPS[0] and task == "coarse" and arm == "0 base":
                    CO[t].setdefault("err0", {})[src] = (node, pred != CO[t]["y"]["coarse"][node])
            LOG.append(dict(rep=REP, block="typing", train=src, marker=task, arm=arm, seconds=time.time() - t0))
        for t in tests:
            for a, b in (("1 +pix", "0 base"), ("2 +pix e2", "0 base"), ("4 +band +pix", "3 +band"),
                         ("3 +band", "0 base"), ("5 +B2", "0 base")):
                lo, hi = ci(BOOT[(a, t)][1] - BOOT[(b, t)][1])
                TYP_DIFF.append(dict(rep=REP, task=task, train=src, test=t, arm=a, base=b,
                                     diff=BOOT[(a, t)][0] - BOOT[(b, t)][0], lo=lo, hi=hi))
        EST[(task, src)] = (time.time() - t_u) / 60
        tick(f"rep {REP} typing {task}, train {src}: " +
             ", ".join(f"{a} {BOOT[(a, src)][0]:.3f}" for a in TYP_ARMS))
        pd.DataFrame(TYP_ROWS).to_csv(f"{OUT}/typing_scores.csv", index=False)
        pd.DataFrame(TYP_DIFF).to_csv(f"{OUT}/typing_diffs.csv", index=False)
        pd.DataFrame(TYP_PCL).to_csv(f"{OUT}/typing_perclass.csv", index=False)
    del PR
    gc.collect()
    pd.DataFrame(LOG).to_csv(f"{OUT}/training_log.csv", index=False)
    EST["rep"] = (time.time() - t_rep) / 60
    tick(f"replicate {REP} done ({EST['rep']:.0f} min)")

## B0 (continued): are typing errors concentrated in cells whose neighbours are mostly a different type?

In [ ]:
ER = []
for t, C in CO.items():
    for src, (node, err) in C.get("err0", {}).items():
        for gname, share in C["share"].items():
            s = share[node]
            bins = pd.cut(s, [-0.001, 0.25, 0.5, 0.75, 1.0], labels=["0–0.25", "0.25–0.5", "0.5–0.75", "0.75–1"])
            df = pd.DataFrame(dict(bin=bins, err=err, cls=np.array(COARSE)[C["y"]["coarse"][node]]))
            for (b, c), gdf in df.dropna().groupby(["bin", "cls"], observed=True):
                ER.append(dict(train=src, test=t, graph=gname, share_bin=str(b), cls=c, n=len(gdf), error_rate=float(gdf.err.mean())))
ER = pd.DataFrame(ER)
ER.to_csv(f"{OUT}/b0_error_by_share.csv", index=False)
if len(ER):
    print(ER[ER.graph == "contact"].pivot_table(index=["train", "test", "cls"], columns="share_bin", values="error_rate").round(3).to_string())

## Decisions (rules fixed in §18.10 and in the first cell)

In [ ]:
ID = pd.DataFrame(IMP_DIFF)
TD = pd.DataFrame(TYP_DIFF)
n_reps = len(set(ID.rep)) if len(ID) else 0
SETTINGS = [("Jackson", "Jackson"), ("Danenberg", "Danenberg"), ("Jackson", "Danenberg"), ("Danenberg", "Jackson")]
lab = lambda s, t: f"{s[0]}->{t[0]}"

def all_pos(df):
    return len(df) == n_reps and n_reps == 3 and bool((df.lo > 0).all())

def imp_pass(arm, src, tst, need_controls):
    hits = []
    for m in MARKERS:
        if m == "estrogen_receptor_alpha":
            continue
        q = ID[(ID.train == src) & (ID.test == tst) & (ID.marker == m)]
        a = q[(q.arm == arm) & (q.base == ("3 +band" if arm == "4 +band +pix" else "0 base"))]
        ok = all_pos(a) and a["diff"].mean() >= 0.02
        if ok and need_controls:
            ok = all_pos(q[(q.arm == "2 +pix e2") & (q.base == "0 base")]) and all_pos(q[(q.arm == "4 +band +pix") & (q.base == "3 +band")])
        if ok:
            hits.append(m)
    return hits

def typ_pass(arm, src, tst, need_controls, task="coarse"):
    q = TD[(TD.task == task) & (TD.train == src) & (TD.test == tst)]
    a = q[(q.arm == arm) & (q.base == "0 base")]
    ok = all_pos(a) and a["diff"].mean() >= 0.01
    if ok and need_controls:
        ok = all_pos(q[(q.arm == "2 +pix e2") & (q.base == "0 base")]) and all_pos(q[(q.arm == "4 +band +pix") & (q.base == "3 +band")])
    return bool(ok)

dec = {"n_reps": n_reps, "B1": {}, "B2": {}}
for s, t in SETTINGS:
    dec["B1"][lab(s, t)] = dict(imputation_markers=imp_pass("1 +pix", s, t, True),
                                coarse_typing=typ_pass("1 +pix", s, t, True) if len(TD) else None)
    dec["B2"][lab(s, t)] = dict(imputation_markers=imp_pass("5 +B2", s, t, False),
                                coarse_typing=typ_pass("5 +B2", s, t, False) if (len(TD) and s != t) else None)
for task, s in (("fine", "Danenberg"), ("fine", "Jackson")):
    dec["B1"][f"fine {lab(s, s)}"] = dict(typing=typ_pass("1 +pix", s, s, True, task) if len(TD) else None)
b1 = any(len(v.get("imputation_markers", [])) >= 2 or v.get("coarse_typing") or v.get("typing") for v in dec["B1"].values())
b2 = any(len(v.get("imputation_markers", [])) >= 2 or v.get("coarse_typing") for v in dec["B2"].values())
dec["B1_pass"], dec["B2_pass"], dec["build_B3"] = bool(b1), bool(b2), bool(b1 or b2)
dec["note"] = "Passes need 3/3 replicates; a smoke or partial run cannot pass."
json.dump(dec, open(f"{OUT}/decision.json", "w"), indent=1)
print(json.dumps(dec, indent=1))

if len(ID):
    s = ID[ID.marker != "estrogen_receptor_alpha"].groupby(["train", "test", "arm", "base"]).agg(
        mean_diff=("diff", "mean"), n_ci_pos=("lo", lambda v: int((v > 0).sum())), n_ci_neg=("hi", lambda v: int((v < 0).sum())))
    print(s.round(4).to_string())
    s.to_csv(f"{OUT}/imputation_diff_summary.csv")
if len(TD):
    s = TD.groupby(["task", "train", "test", "arm", "base"]).agg(
        mean_diff=("diff", "mean"), n_ci_pos=("lo", lambda v: int((v > 0).sum())), n_ci_neg=("hi", lambda v: int((v < 0).sum())))
    print(s.round(4).to_string())
    s.to_csv(f"{OUT}/typing_diff_summary.csv")
json.dump({"smoke": SMOKE, "reps": REPS, "skipped": [list(map(str, x)) for x in SKIPPED], "minutes": (time.time() - T0) / 60},
          open(f"{OUT}/run_info.json", "w"), indent=1)